# SpaceJEPA v3 · 0. The test set

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rk0604/ReEntryAI/blob/jepa-v3/StochasticEntrySim/ReEntryAI_JEPA_v3_0_testset.ipynb)

v3 is a full study of v1 vs v2 vs the classical predictor-corrector (`jepa/v3_plan.md`). This notebook
freezes its **64 entry cases** to Drive and draws their coverage. It needs no GPU (~2 min).

* **16 bridge cases**: the frozen v1/v2 test entries, to connect with earlier results.
* **48 new cases** from a scrambled Sobol sequence (seed 20261018), varying entry altitude (105–135 km),
  speed (7.2–8.4 km/s), flight-path angle (−1.5° to −4°), approach heading (0–360°), and where the
  target sits inside the reachable region. The **target stays at 15°N 15°E**; each entry is placed
  backwards from it.
* Every new target is reachable in all **4 atmospheres** at **≤ 4.5 g** with ≥ 50 km to spare. The
  check uses constant-bank sweeps of the simulator's own equations of motion (no controller is
  involved), restricted to banks whose peak load stays within the 4.5 g study limit.

The set was built once by `python jepa/testset_v3.py` and committed as `configs/v3_testset.csv`.
Here it is copied to Drive and spot-checked against the simulator.

## 0. Setup

On Colab this clones the `jepa-v3` branch, installs `pymsis`, and mounts Google Drive. Every output
goes to **`MyDrive/ReEntryAI_jepa_v3/`** (`ReEntryAI_jepa_v3_smoke/` in smoke mode); the layout is in
`jepa/v3_plan.md` §9 and in the auto-written `00_study/README.md`. The 6,000 flights are read from
`MyDrive/ReEntryAI_jepa_data/wide_6000/`. Every stage is resumable: after a disconnect, run all
cells again and finished work is skipped.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "jepa-v3"
MODE = os.environ.get("JEPA_V3_MODE", "full")     # "full": the study | "smoke": every stage, tiny, numbers meaningless
if IN_COLAB:
    REPO = "/content/ReEntryAI"
    if not os.path.exists(REPO):
        !git clone -q -b {BRANCH} https://github.com/rk0604/ReEntryAI.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q pymsis
    os.chdir(f"{REPO}/StochasticEntrySim")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    STUDY_ROOT = f"{DRIVE}/ReEntryAI_jepa_v3" + ("" if MODE == "full" else f"_{MODE}")
    DATA_CANDIDATES = [f"{DRIVE}/ReEntryAI_jepa_data/wide_6000/episodes_wide_6000.pkl",
                       f"{DRIVE}/ReEntryAI_jepa_data/wide_6000"]
    LOCAL_DATA = "/content/data"
else:
    STUDY_ROOT = os.environ.get("JEPA_V3_ROOT", f"jepa_runs/v3_{MODE}")
    DATA_CANDIDATES = [os.environ.get("JEPA_WIDE_DATA", ""), "jepa_runs/episodes_wide_6000.pkl", "jepa_runs/wide_6000"]
    LOCAL_DATA = "jepa_runs"

import time
from pathlib import Path
SIM_DIR = Path.cwd()
assert (SIM_DIR / "rl_env.py").exists(), "start Jupyter in StochasticEntrySim/"
sys.path.insert(0, str(SIM_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline
from jepa import testset_v3 as TS, study_v3 as S, fly_v3 as F, analyze_v3 as A

CFG = S.StudyConfig.smoke() if MODE == "smoke" else S.StudyConfig()
L = S.Layout(STUDY_ROOT).make()
FROZEN = S.freeze_config(CFG, L)
CPUS = os.cpu_count() or 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"MODE={MODE}  study folder={STUDY_ROOT}  device={DEVICE}  cpus={CPUS}  git={FROZEN['git_commit'][:10]}")

# ---- Auto-disconnect: stop paying for the runtime once the work is done or a cell fails ----
AUTO_DISCONNECT = IN_COLAB      # set to False to keep the runtime (e.g. to explore results by hand)

def release_runtime(reason):
    # Flush Drive, then release the Colab runtime so it stops using compute units.
    if not (IN_COLAB and AUTO_DISCONNECT):
        print(f"[auto-disconnect off] {reason}")
        return
    print(f"[auto-disconnect] {reason}. Flushing Drive and releasing the runtime; outputs stay visible.", flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print("  Drive flush:", e)
    from google.colab import runtime
    runtime.unassign()

def _release_on_error(result):
    err = result.error_in_exec or result.error_before_exec
    if err is not None and not isinstance(err, KeyboardInterrupt):     # pressing stop is not a failure
        release_runtime(f"a cell failed ({type(err).__name__}: {str(err)[:200]})")

_ip = get_ipython()
for _cb in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_release_on_error":
        _ip.events.unregister("post_run_cell", _cb)
_ip.events.register("post_run_cell", _release_on_error)
print("auto-disconnect on: the runtime is released when the last cell finishes or a cell fails" if AUTO_DISCONNECT
      else "auto-disconnect off (not on Colab, or switched off): the runtime stays up until you disconnect it")

## 1. Freeze the committed test set to Drive

In [ ]:
for src, dst in (("configs/v3_testset.csv", "v3_testset.csv"), ("configs/v3_testset_footprints.csv", "footprints.csv"),
                 ("configs/v3_testset_spec.json", "testset_spec.json")):
    data = (SIM_DIR / src).read_bytes()
    if not (L.testset / dst).exists() or (L.testset / dst).read_bytes() != data:
        S.atomic_bytes(L.testset / dst, data)
cases = pd.read_csv(L.testset / "v3_testset.csv")
fps = pd.read_csv(L.testset / "footprints.csv")
spec = __import__("json").loads((L.testset / "testset_spec.json").read_text())
print(f"{len(cases)} cases ({(cases.set == 'new').sum()} new from {spec['n_candidates']} Sobol candidates; "
      f"rejected: {spec['rejections']}); MPPI horizon {spec['horizon_steps']} steps "
      f"({spec['horizon_steps'] * 5} s)")
display(cases[cases.set == "new"][["altitude_m", "velocity_mps", "gamma_deg", "chi_deg", "phi_deg", "lambda_deg",
                                   "downrange_km", "crossrange_km", "approach_deg"]].describe()
        .loc[["min", "50%", "max"]].round(2))

## 2. Feasibility per atmosphere

`feasible_<atmosphere>` = the target is inside that atmosphere's constant-bank footprint within 4.5 g.
Mixing banks reaches at least that region, so this is conservative. The new cases are feasible
everywhere by construction. Some bridge cases are not, e.g. 5 and 14 in the ×0.78 atmosphere,
which v2 also could not reach. Those conditions stay in the primary results; table T02 repeats every
metric without them.

In [ ]:
feas = cases.groupby("set")[[f"feasible_{t}" for t in TS.ATMOSPHERES]].mean().mul(100).round(1)
display(feas)
for t in TS.ATMOSPHERES:
    bad = cases.case[~cases[f"feasible_{t}"]].tolist()
    print(f"{t:>14}: infeasible cases {bad}  max lift-up peak g {cases[f'liftup_peak_g_{t}'].max():.2f}")

## 3. Spot check: the footprint sweep against real simulator flights

In [ ]:
import math, rl_env
from jepa import data as D
params = {t: TS.truth_params(d) for t, d in TS.ATMOSPHERES.items()}
r = cases[cases.set == "new"].iloc[0]
Q = TS.entry_frame(*np.radians([r.phi_deg, r.lambda_deg, r.chi_deg]))
fp = TS.footprint(r.altitude_m, r.velocity_mps, math.radians(r.gamma_deg), params["nominal"])
for sdeg in (0.0, 30.0, -30.0):
    env = rl_env.OrionEntryEnv(controller_mode="policy", dispersion=False, fast_atmosphere=True)
    env.reset(seed=0, options={"ic_override": TS.ic_of(r)})
    done = False
    while not done:
        *_, te, tr, info = env.step(np.array([sdeg / 90.0], dtype=np.float32)); done = te or tr
    dr, cr = TS.to_local(Q, env.x[1], env.x[2])
    p = next(q for q in fp if q["sigma_deg"] == sdeg)
    print(f"bank {sdeg:+5.0f} deg: env lands at {dr:7.1f} / {cr:+6.1f} km, sweep says {p['downrange_km']:7.1f} / "
          f"{p['crossrange_km']:+6.1f} km (peak g {info['peak_g']:.2f} vs {p['peak_g']:.2f})")

## 4. Coverage figures (F01, F02)

In [ ]:
for p in A.testset_figures(L, cases, fps):
    display(Image(str(p)))
S.update_manifest(L)
print("saved to", L.testset)

## Done: release the runtime

Everything this notebook made is already on Drive. This cell flushes Drive and releases the Colab
runtime, so it stops using compute units (a failed cell does the same). To keep the runtime, set
`AUTO_DISCONNECT = False` in the setup cell before running.

In [ ]:
release_runtime("all cells finished")